# SpaceX Falcon 9 Data Wrangling
This notebook cleans raw launch data, imputes missing payload values, categorizes landing outcomes, and constructs the binary classification target `Class`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/dataset_part_1.csv')
print('Loaded dataset shape:', df.shape)
df.head(5)

Loaded dataset shape: (90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 1. Missing Value Analysis & Imputation
Check null values in the dataset and impute missing `PayloadMass` values with the column mean.

In [2]:
print('Null values count:')
print(df.isnull().sum())

mean_payload = df['PayloadMass'].mean()
df['PayloadMass'] = df['PayloadMass'].fillna(mean_payload)
print(f'\nImputed missing PayloadMass with mean: {mean_payload:.2f} kg')
print('Remaining nulls in PayloadMass:', df['PayloadMass'].isnull().sum())

Null values count:
FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

Imputed missing PayloadMass with mean: 6104.96 kg
Remaining nulls in PayloadMass: 0


## 2. Launch Site and Orbit Distribution
Analyze mission counts by launch complex and orbital target.

In [3]:
print('Launch Site Distribution:')
print(df['LaunchSite'].value_counts())

print('\nOrbit Target Distribution:')
print(df['Orbit'].value_counts())

Launch Site Distribution:
LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

Orbit Target Distribution:
Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
HEO       1
ES-L1     1
SO        1
GEO       1
Name: count, dtype: int64


## 3. Landing Outcome Categorization
The `Outcome` column records landing type and success status. We map successful recoveries (`True Ocean`, `True RTLS`, `True ASDS`) to 1, and all failures/unattempted recoveries to 0.

In [4]:
landing_outcomes = df['Outcome'].value_counts()
print('Outcome Frequency Breakdown:')
print(landing_outcomes)

successful_outcomes = {'True ASDS', 'True RTLS', 'True Ocean'}
landing_class = [1 if outcome in successful_outcomes else 0 for outcome in df['Outcome']]
df['Class'] = landing_class

print(f'\nOverall Landing Success Rate: {df["Class"].mean() * 100:.2f}%')
print(df['Class'].value_counts())

Outcome Frequency Breakdown:
Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

Overall Landing Success Rate: 66.67%
Class
1    60
0    30
Name: count, dtype: int64


## 4. Export Cleaned Dataset
Verify and save as `dataset_part_2.csv`.

In [5]:
df.to_csv('../data/dataset_part_2.csv', index=False)
print('Saved dataset_part_2.csv successfully. Shape:', df.shape)

Saved dataset_part_2.csv successfully. Shape: (90, 18)
